# Colombia Open — Lesiones personales

Este notebook muestra cómo cargar el dataset **`delincuencia_lesiones_personales`** (víctimas de lesiones personales y en accidente de tránsito por municipio y día, 2010 a la fecha, Policía Nacional) con `colombia-open`.

> **Granularidad:** una fila por fecha + municipio + conducta + arma/medio + género + grupo etario, con el número de víctimas (~1,377,323 filas). **Para contar víctimas, suma `cantidad` — nunca cuentes filas.**

> **Ojo:** el dataset mezcla lesiones **dolosas** (riñas, agresiones) con lesiones **culposas en accidente de tránsito** (~un tercio de las víctimas). Filtra por `tipo_lesion` antes de hablar de violencia.

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.6.0`.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
lesiones = hub.load_polars("delincuencia_lesiones_personales")

print(lesiones.shape)
lesiones.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | date | Fecha del hecho. |
| `anio` | int | Año de `fecha` — para joins con datasets anuales como `demografia`. |
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Códigos DIVIPOLA — join key hacia `municipios`. La fuente trae un código de 8 dígitos (municipio + centro poblado) que se colapsa a municipio. |
| `nombre_dpto` / `nombre_mpio` | string | Nombres como los entrega la fuente (sin el sufijo "(CT)" de las capitales). |
| `conducta` | string | Artículo del Código Penal tal como lo reporta la fuente. |
| `tipo_lesion` | string | Derivado: `dolosa`, `culposa_transito` o `culposa_otra`. |
| `arma_medio` | string | Arma o medio empleado ("CONTUNDENTES", "VEHICULO", "ARMA DE FUEGO"…). |
| `genero` / `grupo_etario` | string | De la víctima. Pueden ser nulos si la fuente no los reporta. |
| `cantidad` | int | Número de víctimas para esa combinación (siempre > 0). |

Clave primaria: `(fecha, cod_mpio, conducta, arma_medio, genero, grupo_etario)`. Esquema completo: [`docs/datasets/delincuencia_lesiones_personales.md`](../../docs/datasets/delincuencia_lesiones_personales.md).

## Dolosas vs. tránsito, por año

In [ ]:
(
    lesiones.group_by(["anio", "tipo_lesion"])
    .agg(pl.col("cantidad").sum().alias("victimas"))
    .pivot(on="tipo_lesion", index="anio", values="victimas")
    .sort("anio")
)

## Lesiones dolosas: arma/medio y género (nacional, 2025)

In [ ]:
(
    lesiones.filter((pl.col("anio") == 2025) & (pl.col("tipo_lesion") == "dolosa"))
    .group_by(["arma_medio", "genero"])
    .agg(pl.col("cantidad").sum().alias("victimas"))
    .sort("victimas", descending=True)
    .head(10)
)

## Tasa de lesiones dolosas por 100 mil habitantes (join con `demografia`)

`demografia` (proyecciones DANE) cubre 2018-2042, así que la tasa se puede calcular desde 2018. Se usa el área geográfica `"Total"` y se cruza por `(cod_mpio, anio)`.

In [ ]:
poblacion = hub.load_polars("demografia").filter(pl.col("area_geografica") == "Total")

tasas = (
    lesiones.filter(pl.col("tipo_lesion") == "dolosa")
    .group_by(["cod_mpio", "anio"])
    .agg(pl.col("cantidad").sum().alias("victimas"))
    .join(poblacion.select(["cod_mpio", "nombre_mpio", "anio", "poblacion"]), on=["cod_mpio", "anio"])
    .with_columns((pl.col("victimas") / pl.col("poblacion") * 100_000).round(1).alias("tasa_100k"))
)

# Municipios de más de 500 mil habitantes con mayor tasa en 2025
tasas.filter((pl.col("anio") == 2025) & (pl.col("poblacion") > 500_000)).sort(
    "tasa_100k", descending=True
).head(10)

## Warnings de validación y procedencia

El validador reporta como **warning** (no error) cualquier `cod_mpio` que no exista en el `municipios` vigente, y cualquier conducta nueva de la fuente que el pipeline todavía no clasifique en `tipo_lesion`. La metadata de procedencia queda en el catálogo:

In [ ]:
for warning in hub.get_dataset("delincuencia_lesiones_personales")["warnings"]:
    print("warning:", warning)

hub.provenance("delincuencia_lesiones_personales")

## Limitaciones conocidas

- **Mezcla delitos y accidentes**: filtra por `tipo_lesion` según la pregunta.
- **4 filas sin municipio excluidas** (`codigo_dane = "00000000"` en la fuente) y **~25 filas con código y nombre contradictorios** — se confía en el código; el nombre publicado es el más frecuente para ese código.
- **Centro poblado no publicado**: los 3 últimos dígitos del código de la fuente se descartan al agregar a municipio.
- **La serie no es homogénea en el tiempo** (cambios de registro y denuncia; 2020 cae por la pandemia) y **el año en curso está incompleto** (corte mensual).

Documentación completa: [`docs/datasets/delincuencia_lesiones_personales.md`](../../docs/datasets/delincuencia_lesiones_personales.md).